# Phase 4: End-to-End Trust & Safety Pipeline
This notebook demonstrates the complete, production-ready pipeline. 
1. We ingest a batch of mixed (Fake + Real) reviews.
2. The Dual-Input **RoBERTa + Stylometry** classifier detects and drops the fakes.
3. The remaining genuine reviews are passed to **BART** to generate a trustworthy summary for the buyer.

In [ ]:
import pandas as pd
import torch
from transformers import pipeline

# 1. Load the Incoming Batch of Reviews
print("--- STAGE 1: INGESTION ---")
df = pd.read_csv("../data/processed/reviews_with_stylometry.csv")
print(f"Total reviews in batch: {len(df)}")
df[['text', 'is_fake']].head(10)

## The Classifier Stage
Normally, we would pass these through our loaded `fraud_detector_lora.pt` model. 
Since this notebook is a visual demonstration, we will simulate the classification by dropping the known fakes.

In [ ]:
print("--- STAGE 2: FRAUD FILTERING ---")
genuine_reviews = df[df['is_fake'] == 0]
print(f"Detected and dropped {len(df) - len(genuine_reviews)} Fake Reviews!")
print(f"Passing {len(genuine_reviews)} Genuine Reviews to the Generative AI...")

## The Generative AI Summarization Stage
Now we use BART to synthesize the genuine reviews.

In [ ]:
print("--- STAGE 3: GENERATIVE SUMMARIZATION ---")
# Taking the first 15 genuine reviews for demonstration
text_to_summarize = " ".join(genuine_reviews['text'].head(15).tolist())

# Load BART (This may take a minute on CPU)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Generate the summary
summary = summarizer(text_to_summarize, max_length=130, min_length=30, do_sample=False)

print("\n" + "="*50)
print("FINAL BUYER SUMMARY (Filtered of Fake Reviews):")
print("="*50)
print(summary[0]['summary_text'])
print("="*50)